# Tutorial 5 — Optimize, reuse, and resolve the IPF model

## Prerequisites

Complete Tutorials 1–4. In particular, Tutorial 4 established the IPF
topology, public refs, typed views, and model-owned default recipe.

## Objects introduced here

Immutable optimization overrides, request-local `ParameterSet`
extrapolation, `PumpAxis`, `CurrentDrive`, `HBSolveSpec`, `HBCaseSpec`,
`HBTruncation`, typed `HBBatchResult` cases, reports, and exact
`resolve()`.

## What the reader will build

One sealed model request: inspect the default, optionally make an
immutable consumer override, optimize it, reuse its `ParameterSet` for
Direct and pump-off HB responses, assemble a report, and resolve that
exact optimization after restarting a kernel.

## What inspection or Result is produced

After the runtime is implemented, this workflow returns one optimization
result, one Direct response, one named pump-off HB case, and one report.
The current `CONVERGING` scaffold is fail-fast and this documentation
does not execute cells while rendering.

## Import the model, then the terminal workflow

The consumer first needs the team’s target and model façade; the
terminal request helpers are imported separately so construction stays
inspectable.

In [ ]:
from circuit_model import IPFTarget, build_model, build_session
from scnsim import units as u

## Bind the consumer-owned target

The target and workspace are request inputs. They do not restate RLGC,
topology, selectors, bounds, weights, or optimizer controls, and they
are not acceptance Gates.

In [ ]:
target = IPFTarget(
    readout_frequency=6.2 * u.GHz,
    filter_frequency=7.1 * u.GHz,
    transfer_zero_frequency=6.6 * u.GHz,
    coupling=45.0 * u.MHz,
    combined_linewidth=2.0 * u.MHz,
    response_frequencies=tuple((5.0 + 0.02 * i) * u.GHz for i in range(151)),
)
workspace = "workspace/synthetic_ipf"
model = build_model()
session = build_session(model, workspace=workspace)

## Inspect the default before deciding to depart from it

The model default is the normal consumer path. `show()` and
`run.explain()` make its variables, Direct quantity objectives,
normalized costs, and exact bound request visible without executing a
search.

In [ ]:
default_spec = model.build_default_optimization_spec(target)
default_spec.show()
default_spec.variable(model.idc_finger_length).bounds
session.run.explain(session.optimization_view, default_spec).show()

## Make an immutable override only when the request needs one

The pain with changing a model default in place is that another notebook
can no longer reconstruct what ran. `with_variable_overrides()` instead
returns a new spec keyed by the exact public `ParameterRef`. This
override authorizes IDC extrapolation only for this optimization
request; it changes neither the model default nor later Direct/HB
requests.

In [ ]:
custom_spec = default_spec.with_variable_overrides(
    bounds={
        model.idc_finger_length: (38.0 * u.um, 76.0 * u.um),
    },
    allow_extrapolation=(model.idc_finger_length,),
)
custom_spec.show()
request_spec = custom_spec

## Optimize once and retain the winner

This walkthrough selects the immutable custom request above.
`run.optimize()` is Direct-only; it returns an immutable result rather
than storing a mutable “current winner” on the session.
`best.parameters` is the exact `ParameterSet` that later requests must
receive explicitly.

In [ ]:
optimization = session.run.optimize(session.optimization_view, request_spec)
winner = optimization.best.parameters

The optimization’s extrapolation approval does not leak into another
request. `ParameterSet` creates the explicit response-request variant
and repeats the approval for the same public parameter. If the winner
remains inside support, the same construction is still an honest no-op
approval record.

In [ ]:
from scnsim import ParameterSet

response_parameters = ParameterSet(
    values=winner.values,
    allow_extrapolation=(model.idc_finger_length,),
)

## Request winner-only Direct and pump-off HB responses

The workflow helper declares one Direct S trace plus a pump-ready HB
request. `PumpAxis` and `CurrentDrive` establish the possible pump
basis; the named `HBCaseSpec(id="pump_off", currents={})` leaves that
drive off. Its `HBSolveSpec` still records the selected truncation, so a
pump-off comparison has an explicit basis rather than a hidden linear
fallback. This comparison also declares both 3WM and 4WM disabled;
enabling either is a different sealed request. The 9 GHz axis does not
create a zero-frequency response mode on the 5–8 GHz signal grid. The
Direct trace uses empty mode tuples, while the HB trace uses `(0,)` to
address the signal mode on the one-axis lattice. `run.solve()` returns
an `HBBatchResult`; `hb.cases[id]` is the only public lookup for one
declared case, so the code selects `"pump_off"` explicitly.

In [ ]:
from workflow import build_response_specs

direct_spec, hb_spec = build_response_specs(model, target)
direct = session.run.solve(
    session.response_view,
    direct_spec,
    parameters=response_parameters,
)
hb = session.run.solve(
    session.response_view,
    hb_spec,
    parameters=response_parameters,
)
pump_off = hb.cases["pump_off"]

## Inspect the parallel selected-network surfaces

Both Results expose S, Y, and Z on the same selected response view. S
has presentation choices; Y and Z expose their labeled matrix views
directly. The named case lookup is intentional: there is no implicit
“pump result.”

In [ ]:
direct.s.show(magnitude="linear")
direct.s.show(magnitude="db")
direct.y.view
direct.z.view
pump_off.s.show(magnitude="linear")
pump_off.s.show(magnitude="db")
pump_off.y.view
pump_off.z.view

## Assemble the report from existing Results

`build_report()` consumes exact materialized inputs. It does not solve
again, select a latest result, or reinterpret the target.

In [ ]:
from scnsim import ReportSpec

report = session.run.build_report(
    ReportSpec(inputs=(optimization, direct, pump_off))
)
report.show()

## Resolve the exact optimization after restart

After a kernel restart there is no trustworthy in-memory target, model,
Ref, or Spec. The cell below is deliberately self-contained: it rebuilds
the same target and workspace, recreates the same custom override
against the rebuilt model’s `ParameterRef`, and asks that rebuilt Run to
resolve the exact request. `resolve()` verifies and loads it; it never
reruns optimization or guesses a latest workspace result.

In [ ]:
from circuit_model import IPFTarget, build_model, build_session
from scnsim import units as u

restart_target = IPFTarget(
    readout_frequency=6.2 * u.GHz,
    filter_frequency=7.1 * u.GHz,
    transfer_zero_frequency=6.6 * u.GHz,
    coupling=45.0 * u.MHz,
    combined_linewidth=2.0 * u.MHz,
    response_frequencies=tuple(
        (5.0 + 0.02 * i) * u.GHz for i in range(151)
    ),
)
restart_workspace = "workspace/synthetic_ipf"
restart_model = build_model()
restart_session = build_session(restart_model, workspace=restart_workspace)
restart_default = restart_model.build_default_optimization_spec(restart_target)
restart_spec = restart_default.with_variable_overrides(
    bounds={
        restart_model.idc_finger_length: (38.0 * u.um, 76.0 * u.um),
    },
    allow_extrapolation=(restart_model.idc_finger_length,),
)
resolved = restart_session.run.resolve(
    restart_session.optimization_view,
    restart_spec,
)
resolved.show()

## Reusable terminal-workflow source

The team façade packages the model-default optimize → winner response →
report sequence for a consumer that wants one call. Its default resolver
matches that default request. This tutorial rebuilt its custom request
directly because exact resolve must preserve the override. The source
remains separate from model construction and is included directly, not
copied into this tutorial.

``` python
"""Terminal workflow for the synthetic SCNSim IPF optimization example.

This module owns requests that execute: default optimization, winner-only
Direct and pump-off HB responses, reporting, and exact receipt resolution.
Model construction remains in :mod:`circuit_model` so model inspection never
imports a terminal workflow.
"""

from __future__ import annotations

from dataclasses import dataclass
from os import PathLike

from circuit_model import IPFModel, IPFSession, IPFTarget, build_model, build_session
from scnsim import (
    CurrentDrive,
    DirectSolveResult,
    DirectSolveSpec,
    HBCaseSpec,
    HBBatchResult,
    HBSolveSpec,
    HBTruncation,
    OptimizationResult,
    PumpAxis,
    ReportResult,
    ReportSpec,
    SParameterTrace,
    units as u,
)


@dataclass(frozen=True)
class WorkflowResult:
    """Exact immutable Results from one model-owned optimization workflow."""

    optimization: OptimizationResult
    direct: DirectSolveResult
    hb: HBBatchResult
    report: ReportResult


def build_response_specs(
    model: IPFModel,
    target: IPFTarget,
) -> tuple[DirectSolveSpec, HBSolveSpec]:
    """Build winner-only Direct and pump-off HB response requests."""

    direct_trace = SParameterTrace(
        id="transmission",
        input_port="feedline_in",
        input_mode=(),
        output_port="feedline_out",
        output_mode=(),
    )
    direct = DirectSolveSpec(
        frequencies=target.response_frequencies,
        traces=(direct_trace,),
    )
    pump_axis = PumpAxis(id="pump", frequency=9.0 * u.GHz)
    pump_drive = CurrentDrive(
        id="pump_drive",
        at=model.feedline_in_port,
        mode=(1,),
    )
    hb_trace = SParameterTrace(
        id="transmission",
        input_port="feedline_in",
        input_mode=(0,),
        output_port="feedline_out",
        output_mode=(0,),
    )
    hb = HBSolveSpec(
        pump_axes=(pump_axis,),
        drives=(pump_drive,),
        frequencies=target.response_frequencies,
        cases=(HBCaseSpec(id="pump_off", currents={}),),
        truncation=HBTruncation(
            pump_harmonics=(3,),
            modulation_harmonics=(1,),
            three_wave_mixing=False,
            four_wave_mixing=False,
        ),
        traces=(hb_trace,),
    )
    return direct, hb


def optimize_ipf(
    *,
    target: IPFTarget,
    workspace: str | PathLike[str],
) -> WorkflowResult:
    """Optimize the model default and materialize its winner-only responses."""

    model = build_model()
    session = build_session(model, workspace=workspace)
    optimization = session.run.optimize(
        session.optimization_view,
        model.build_default_optimization_spec(target),
    )
    direct_spec, hb_spec = build_response_specs(model, target)
    direct = session.run.solve(
        session.response_view,
        direct_spec,
        parameters=optimization.best.parameters,
    )
    hb = session.run.solve(
        session.response_view,
        hb_spec,
        parameters=optimization.best.parameters,
    )
    report = session.run.build_report(
        ReportSpec(inputs=(optimization, direct, hb.cases["pump_off"]))
    )
    return WorkflowResult(
        optimization=optimization,
        direct=direct,
        hb=hb,
        report=report,
    )


def resolve_ipf_optimization(
    *,
    target: IPFTarget,
    workspace: str | PathLike[str],
) -> OptimizationResult:
    """Resolve the exact model-default optimization receipt after restart."""

    model = build_model()
    session = build_session(model, workspace=workspace)
    resolved = session.run.resolve(
        session.optimization_view,
        model.build_default_optimization_spec(target),
    )
    if not isinstance(resolved, OptimizationResult):
        raise TypeError("exact request did not resolve to OptimizationResult")
    return resolved
```

## Learned objects and next

You inspected a default, made an immutable request-local override,
optimized once, reused `best.parameters` for Direct and named HB
responses, presented parallel S/Y/Z surfaces, built a report, and
resolved the exact receipt after restart. You can now adapt a
model-owned recipe without taking ownership of its topology or silently
creating a second workflow authority.